# L4 - Working RAG

## Overview

Recall the overall workflow for retrieval augmented generation (RAG):

![overview.jpeg](attachment:overview.jpeg)

In [10]:
!pip install pypdf langchain-community langchain-openai langchain-text-splitters chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 48.2 MB/s  0:00:00 39.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 62.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 61.4 MB/s  0:00:00 69.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23/23 [chromadb] 22/23 [chromadb]s]d]ntime]rotos]


In [5]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loaders = [
    # Duplicate documents on purpose - messy data
    PyPDFLoader("docs/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture02.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture03.pdf")
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 150
)

splits = text_splitter.split_documents(docs)

In [14]:
import requests
from typing import List
from langchain_core.embeddings import Embeddings

class LocalServerEmbeddings(Embeddings):
    def __init__(self, base_url: str):
        self.base_url = base_url
        self.model = "text-embedding-nomic-embed-text-v1.5"

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        response = requests.post(f"{self.base_url}/embeddings", json={"model": self.model, "input": texts})
        data = response.json()

        return [item["embedding"] for item in data["data"]]

    def embed_query(self, text: str) -> List[float]:
        response = requests.post(f"{self.base_url}/embeddings", json={"model": self.model, "input": [text]})
        data = response.json()
        return data["data"][0]["embedding"]

embedding = LocalServerEmbeddings(base_url="http://localhost:1234/v1")

In [16]:
from langchain_community.vectorstores import Chroma

!rm -rf ./docs/chroma

persist_directory = 'chroma/'

vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=persist_directory
)

In [17]:
import os

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

base_url = "http://localhost:1234/v1"
api_key = "lm-studio"
llm_model = "qwen/qwen3.5-9b"

llm = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    temperature=0.9,
    model=llm_model
)

response = llm.invoke([
    HumanMessage(content="What is the capital of France?")
])

print(response.content)



The capital of France is Paris.


In [18]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

persist_directory = 'chroma/'

vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

/var/folders/2y/w5660wqs0rg9jt4tgkf4x4940000gn/T/ipykernel_79147/3331618096.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)


In [19]:
print(vectordb._collection.count())

208


In [20]:
question = "What are the most important topics discussed in this course?"
docs = vectordb.similarity_search(question,k=3)
len(docs)

3

### RetrievalQA chain

In [21]:
from langchain.chains import RetrievalQA

ModuleNotFoundError: No module named 'langchain.chains'

In [34]:
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever()
)

In [38]:
result = qa_chain({"query": question})

In [39]:
result["result"]

"Based on the provided context, some of the key topics and components discussed in this machine learning course include:\n\n1. **Convex Optimization**: This is likely covered both in main lectures and extended discussions during the optional discussion sections.\n\n2. **Hidden Markov Models (HMMs)**: These are types of algorithms used for modeling time series data, which will be briefly discussed in the extension sessions.\n\n3. **Prerequisites Refreshers**: In the initial weeks of discussion sections, there is a focus on covering prerequisites such as probability, statistics, and algebra to help refresh those concepts for students who might need it.\n\n4. **MATLAB Usage**: There's a strong emphasis on learning MATLAB, with specific mention that understanding and using MATLAB will be beneficial and included in the course materials.\n\n5. **Project Work**: The term project can be completed individually or in small groups (up to three people), and while grading doesn't differ by group si

### Prompt

In [40]:
from langchain_core.prompts import PromptTemplate

template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Use three sentences maximum. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer. 
{context}
Question: {question}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(template)


In [41]:
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

In [42]:
question = "Will we learn about LLMs at this course?"

In [43]:
result = qa_chain({"query": question})

In [44]:
result["result"]

"The context provided does not mention learning about Large Language Models (LLMs) in the course. The lectures focus on topics like linear regression, logistic regression, perceptron algorithm, and Newton's method for fitting models. Thanks for asking!"

In [45]:
result["source_documents"]

[Document(metadata={'author': '', 'creationdate': '2008-07-11T11:25:03-07:00', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2008-07-11T11:25:03-07:00', 'page': 0, 'page_label': '1', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'source': 'docs/MachineLearning-Lecture03.pdf', 'title': '', 'total_pages': 16}, page_content='MachineLearning-Lecture03  \nInstructor (Andrew Ng):Okay. Good morning and welcome back to the third lecture of \nthis class. So here’s what I want to do today, and some of the topics I do today may seem \na little bit like I’m jumping, sort of, from topic to topic, but here’s, sort of, the outline for \ntoday and the illogical flow of ideas. In the last lecture, we talked about linear regression \nand today I want to talk about sort of an adaptation of that called locally weighted \nregression. It’s very a popular algorithm that’s actually one of my former mentors \nprobably favorite machine learning algorithm.  \nWe’ll then talk about a probable second int